# Thu Thập & Làm Sạch Dữ Liệu Sách từ Google Books API (6.000 – 12.000 cuốn)

Notebook này được chia thành **6 bước rõ ràng**:
1. **Cấu hình & Import thư viện**
2. **Danh sách 100+ từ khóa đa lĩnh vực**
3. **Hàm cào dữ liệu từ Google Books API** *(kèm nội dung khái quát, phân trang, retry 429)*
4. **Chạy cào dữ liệu thô & hỗ trợ Resume (chạy tiếp nếu dừng giữa chừng)**
5. **Làm sạch dữ liệu (Data Cleaning)**: gỡ thẻ HTML, chuẩn hóa năm/số trang/ảnh bìa, tạo tóm tắt khái quát, khử trùng lặp
6. **Thống kê & Xuất dữ liệu sạch** ra `books_cleaned.csv` và `books_cleaned.json`

## Bước 1: Cài đặt (nếu thiếu) & Cấu hình môi trường

In [ ]:
# Bỏ dấu # ở dòng dưới nếu máy chưa cài requests hoặc pandas
# !pip install requests pandas

import os
import re
import html
import json
import time
import requests
import pandas as pd

# Đọc API Key tự động từ file .env (hoặc dán trực tiếp key của bạn vào biến API_KEY)
API_KEY = ""
if os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip().startswith("VITE_GOOGLE_BOOKS_API_KEY="):
                API_KEY = line.split("=", 1)[1].strip()
                break

# Cấu hình thông số cào dữ liệu
TARGET_BOOKS = 10000         # Mục tiêu số sách (có thể chỉnh 6000 - 12000)
MAX_RESULTS = 40             # Tối đa 40 sách/request theo quy định Google Books
MAX_PAGES_PER_QUERY = 5      # Tối đa 5 trang/từ khóa (~200 sách/từ khóa)

RAW_JSON_FILE = "google_books_raw.json"
STATE_FILE = "crawl_state.json"
CLEAN_CSV_FILE = "books_cleaned.csv"
CLEAN_JSON_FILE = "books_cleaned.json"

print("Đã tải API Key:" , "Có (hợp lệ)" if len(API_KEY) > 10 else "Chưa có (sẽ chạy không key - dễ bị giới hạn)")

: 

## Bước 2: Danh sách 100+ từ khóa chia theo từng nhóm chủ đề

In [ ]:
KEYWORD_GROUPS = {
    "Tâm lý & Chữa lành": [
        "tâm lý học", "chữa lành tâm hồn", "cân bằng cảm xúc", "vượt qua trầm cảm",
        "thiền định chánh niệm", "trí tuệ cảm xúc", "subject:psychology",
        "mental health mindfulness", "cognitive behavioral therapy", "emotional intelligence"
    ],
    "Kỹ năng & Phát triển bản thân": [
        "kỹ năng sống", "phát triển bản thân", "thói quen tốt", "quản lý thời gian",
        "kỹ năng giao tiếp", "tư duy phản biện", "tư duy tích cực", "rèn luyện kỷ luật",
        "đắc nhân tâm", "subject:self-help", "productivity deep work", "atomic habits",
        "critical thinking logic", "communication skills"
    ],
    "Kinh doanh & Tài chính": [
        "kinh doanh", "khởi nghiệp", "quản trị doanh nghiệp", "tài chính cá nhân",
        "đầu tư chứng khoán", "bất động sản", "marketing bán hàng", "thương hiệu",
        "kinh tế học", "đàm phán thương lượng", "subject:business", "subject:economics",
        "subject:finance", "subject:marketing", "subject:management", "subject:leadership",
        "startup entrepreneurship", "personal finance investing", "behavioral economics"
    ],
    "Công nghệ & AI": [
        "công nghệ thông tin", "trí tuệ nhân tạo", "lập trình máy tính", "khoa học dữ liệu",
        "an ninh mạng", "subject:computers", "subject:technology", "artificial intelligence",
        "machine learning", "software engineering", "clean code architecture",
        "data science analytics", "cybersecurity", "cloud computing",
        "web development", "python programming", "javascript typescript", "ui ux design"
    ],
    "Văn học & Tiểu thuyết": [
        "văn học việt nam", "tiểu thuyết hiện đại", "truyện ngắn hay", "trinh thám",
        "khoa học viễn tưởng", "subject:fiction", "subject:literature", "subject:classics",
        "subject:mystery", "subject:science fiction", "subject:fantasy", "biography memoir"
    ],
    "Khoa học, Triết học & Xã hội": [
        "khoa học vũ trụ", "lịch sử việt nam", "lịch sử thế giới", "triết học",
        "xã hội học", "văn hóa việt nam", "subject:philosophy", "subject:history",
        "subject:science", "subject:physics", "subject:biology", "neuroscience brain",
        "world history", "ancient civilizations"
    ],
    "Sức khỏe, Giáo dục & Nghệ thuật": [
        "sức khỏe dinh dưỡng", "y học thường thức", "nuôi dạy con", "giáo dục học",
        "học tiếng anh", "kiến trúc nghệ thuật", "ẩm thực nấu ăn", "subject:health",
        "subject:nutrition", "subject:education", "subject:design", "modern art architecture"
    ]
}

ALL_QUERIES = [(group, kw) for group, kws in KEYWORD_GROUPS.items() for kw in kws]
print(f"Tổng số nhóm chủ đề: {len(KEYWORD_GROUPS)} | Tổng số từ khóa: {len(ALL_QUERIES)}")

## Bước 3: Hàm gọi Google Books API (lấy cả nội dung khái quát & retry lỗi 429)
- Kết hợp lấy `volumeInfo.description` và `searchInfo.textSnippet` (nội dung khái quát hiển thị trên Google Books) để tối đa hóa số sách có tóm tắt nội dung.

In [ ]:
def clean_cover_url(image_links: dict) -> str:
    if not isinstance(image_links, dict):
        return ""
    raw = image_links.get("thumbnail") or image_links.get("smallThumbnail") or ""
    if not raw:
        return ""
    url = re.sub(r"^http://", "https://", raw, flags=re.IGNORECASE)
    url = url.replace("&edge=curl", "")
    url = re.sub(r"zoom=\d", "zoom=1", url)
    return url

def fetch_books_page(query: str, start_index: int, max_results: int = 40, retries: int = 3) -> dict:
    url = "https://www.googleapis.com/books/v1/volumes"
    params = {
        "q": query,
        "maxResults": max_results,
        "startIndex": start_index,
    }
    if API_KEY:
        params["key"] = API_KEY

    for attempt in range(1, retries + 1):
        resp = requests.get(url, params=params, timeout=15)
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code in (429, 500, 502, 503):
            wait_sec = attempt * 3
            print(f"  [HTTP {resp.status_code}] Tạm nghỉ {wait_sec}s rồi thử lại ({attempt}/{retries})...")
            time.sleep(wait_sec)
            continue
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")

    raise RuntimeError("Vượt quá số lần thử lại do Rate Limit")

def parse_book_item(item: dict, topic_group: str, query: str) -> dict | None:
    info = item.get("volumeInfo", {})
    search_info = item.get("searchInfo", {})
    title = (info.get("title") or "").strip()
    if not title:
        return None

    authors = info.get("authors") or ["Chưa rõ tác giả"]
    categories = info.get("categories") or [query.replace("subject:", "")]

    # Lấy nội dung khái quát: ưu tiên description đầy đủ, nếu thiếu thì lấy textSnippet
    raw_description = (info.get("description") or "").strip()
    raw_snippet = (search_info.get("textSnippet") or "").strip()
    overview_raw = raw_description if len(raw_description) >= len(raw_snippet) else raw_snippet

    return {
        "id": item.get("id", ""),
        "title": title,
        "subtitle": (info.get("subtitle") or "").strip(),
        "author": ", ".join(authors),
        "publisher": (info.get("publisher") or "Chưa rõ NXB").strip(),
        "publishDate": (info.get("publishedDate") or "").strip(),
        "pages": info.get("pageCount") or 0,
        "topicGroup": topic_group,
        "genre": " | ".join(categories),
        "rating": info.get("averageRating") or 4.5,
        "ratingsCount": info.get("ratingsCount") or 0,
        "language": info.get("language") or "unknown",
        "coverUrl": clean_cover_url(info.get("imageLinks")),
        "previewLink": info.get("previewLink") or "",
        "description_raw": overview_raw,
        "snippet_raw": raw_snippet,
    }

## Bước 4: Chạy thu thập dữ liệu thô (hỗ trợ Resume)
- Dữ liệu được lưu liên tục sau mỗi từ khóa vào `google_books_raw.json`.
- Nếu dừng giữa chừng và chạy lại cell này, code sẽ tự động bỏ qua các từ khóa đã cào xong.

In [ ]:
books_map = {}
if os.path.exists(RAW_JSON_FILE):
    with open(RAW_JSON_FILE, "r", encoding="utf-8") as f:
        for b in json.load(f):
            key = f"{b['title'].lower()}_{b['author'].split(',')[0].lower().strip()}"
            books_map[key] = b
    print(f"Đã nạp lại {len(books_map)} sách từ file tạm.")

done_queries = set()
if os.path.exists(STATE_FILE):
    with open(STATE_FILE, "r", encoding="utf-8") as f:
        done_queries = set(json.load(f))

for idx, (group, query) in enumerate(ALL_QUERIES, start=1):
    if len(books_map) >= TARGET_BOOKS:
        print(f"\nĐã đạt mục tiêu {TARGET_BOOKS} sách! Dừng thu thập.")
        break
    if query in done_queries:
        continue

    print(f"[{idx}/{len(ALL_QUERIES)}] ({group}) Từ khóa: '{query}' | Hiện có: {len(books_map)}/{TARGET_BOOKS}")
    for page in range(MAX_PAGES_PER_QUERY):
        if len(books_map) >= TARGET_BOOKS:
            break
        try:
            data = fetch_books_page(query, start_index=page * MAX_RESULTS, max_results=MAX_RESULTS)
            items = data.get("items") or []
            if not items:
                break

            for item in items:
                parsed = parse_book_item(item, group, query)
                if not parsed:
                    continue
                dedup_key = f"{parsed['title'].lower()}_{parsed['author'].split(',')[0].lower().strip()}"
                if dedup_key not in books_map:
                    books_map[dedup_key] = parsed

            if len(items) < MAX_RESULTS:
                break
            time.sleep(0.35)
        except Exception as e:
            print(f"  Lỗi tại trang {page + 1}: {e}")
            break

    done_queries.add(query)
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(list(done_queries), f, ensure_ascii=False, indent=2)
    with open(RAW_JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(list(books_map.values()), f, ensure_ascii=False, indent=2)
    time.sleep(0.4)

print(f"\nHoàn tất cào thô! Tổng số sách thu được: {len(books_map)}")

## Bước 5: Làm sạch dữ liệu & Tạo nội dung khái quát (`summary`)
Các xử lý làm sạch được áp dụng:
- **Gỡ thẻ HTML & ký tự mã hóa**: `html.unescape` + xóa `<p>`, `<br>`, `<b>`, `<i>`, `&quot;`...
- **Chuẩn hóa khoảng trắng & ký tự điều khiển**: loại bỏ tab, xuống dòng thừa, ký tự rác.
- **Tạo cột `summary` (Nội dung khái quát)**: trích xuất gọn 2–3 câu đầu (hoặc ~280 ký tự tròn câu) từ mô tả đã làm sạch.
- **Chuẩn hóa kiểu dữ liệu**: trích xuất `publishYear` hợp lệ (1900–2026), ép kiểu `pages`, `rating`, `ratingsCount`.
- **Lọc bỏ bản ghi rỗng/rác & Khử trùng lặp** theo `id` và `(title, author)`.

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    # Giải mã HTML entities (&amp;, &quot;, ...)
    t = html.unescape(text)
    # Thay thẻ xuống dòng bằng dấu cách rồi xóa toàn bộ thẻ HTML
    t = re.sub(r"<br\s*/?>|</p>|</div>", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"<[^>]+>", " ", t)
    # Xóa ký tự điều khiển và khoảng trắng thừa
    t = re.sub(r"[\r\n\t]+", " ", t)
    t = re.sub(r"\s{2,}", " ", t).strip()
    return t

def make_overview_summary(clean_desc: str, max_chars: int = 280) -> str:
    if not clean_desc:
        return ""
    # Tách theo câu và lấy 2-3 câu đầu sao cho súc tích, tròn nghĩa
    sentences = re.split(r"(?<=[.!?])\s+", clean_desc)
    summary = ""
    for s in sentences:
        if len(summary) + len(s) + 1 <= max_chars:
            summary = f"{summary} {s}".strip()
        else:
            break
    if not summary:
        summary = clean_desc[:max_chars].rsplit(" ", 1)[0] + "..."
    return summary

def extract_year(date_str: str) -> int | None:
    if not isinstance(date_str, str):
        return None
    m = re.search(r"\b(19\d{2}|20[0-2]\d)\b", date_str)
    return int(m.group(1)) if m else None

# Đọc dữ liệu thô
with open(RAW_JSON_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f"Số dòng trước khi làm sạch: {len(df)}")

# 1. Làm sạch tiêu đề, tác giả, NXB
df["title"] = df["title"].apply(clean_text)
df["author"] = df["author"].apply(clean_text).replace("", "Chưa rõ tác giả")
df["publisher"] = df["publisher"].apply(clean_text).replace("", "Chưa rõ NXB")

# 2. Làm sạch nội dung mô tả & tạo nội dung khái quát (summary)
df["description"] = df["description_raw"].apply(clean_text)
df["summary"] = df["description"].apply(make_overview_summary)

# 3. Chuẩn hóa năm xuất bản, số trang, điểm đánh giá
df["publishYear"] = df["publishDate"].apply(extract_year)
df["pages"] = pd.to_numeric(df["pages"], errors="coerce").fillna(0).astype(int)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(4.5).clip(0, 5).round(2)
df["ratingsCount"] = pd.to_numeric(df["ratingsCount"], errors="coerce").fillna(0).astype(int)

# 4. Lọc các dòng hợp lệ (có tiêu đề và nội dung khái quát >= 20 ký tự)
# Nếu muốn giữ cả sách không có mô tả, hãy đổi >= 20 thành >= 0
df_clean = df[(df["title"].str.len() >= 2) & (df["description"].str.len() >= 20)].copy()

# 5. Khử trùng lặp theo id và theo (title + author)
df_clean["_title_norm"] = df_clean["title"].str.lower().str.strip()
df_clean["_author_norm"] = df_clean["author"].str.lower().str.split(",").str[0].str.strip()
df_clean = df_clean.drop_duplicates(subset=["id"])
df_clean = df_clean.drop_duplicates(subset=["_title_norm", "_author_norm"])

# Chọn các cột sạch cuối cùng
final_cols = [
    "id", "title", "subtitle", "author", "publisher", "publishYear",
    "pages", "topicGroup", "genre", "rating", "ratingsCount", "language",
    "coverUrl", "previewLink", "summary", "description"
]
df_clean = df_clean[final_cols].reset_index(drop=True)
print(f"Số dòng sau khi làm sạch & khử trùng: {len(df_clean)}")
df_clean.head(5)

## Bước 6: Thống kê chất lượng dữ liệu & Xuất file sạch

In [ ]:
print("=== THỐNG KÊ DỮ LIỆU SẠCH ===")
print("Tổng số sách:", len(df_clean))
print("Số sách có ảnh bìa:", (df_clean["coverUrl"].str.len() > 0).sum())
print("Số sách có nội dung khái quát (summary):", (df_clean["summary"].str.len() > 0).sum())
print("\nPhân bố theo nhóm chủ đề:")
print(df_clean["topicGroup"].value_counts())
print("\nTop 10 ngôn ngữ:")
print(df_clean["language"].value_counts().head(10))

# Xuất ra CSV (UTF-8 BOM để mở Excel không lỗi tiếng Việt) và JSON
df_clean.to_csv(CLEAN_CSV_FILE, index=False, encoding="utf-8-sig")
df_clean.to_json(CLEAN_JSON_FILE, orient="records", force_ascii=False, indent=2)

print(f"\nĐã lưu thành công: {CLEAN_CSV_FILE} và {CLEAN_JSON_FILE}")